# 03 - Operações DML em Tabelas Delta Lake

Demonstra as três operações DML (**INSERT**, **UPDATE**, **DELETE**) em tabelas Delta Lake armazenadas no MinIO, além de **HISTORY** (histórico de versões) e **TIME TRAVEL** (leitura de versões anteriores).

As operações são realizadas via **Spark SQL** e também via **DeltaTable API** (Python).

**Pré-requisitos:** Notebook `02` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração e SparkSession

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('DML_Delta_LojoDB')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')
spark

## 2. Registrar Tabelas Delta como SQL Views

Duas abordagens para registrar tabelas Delta:
- **Tabela não gerenciada (external):** `CREATE TABLE ... LOCATION '...'` — metadados no Spark, dados no MinIO. Ao executar `DROP TABLE`, apenas os metadados são removidos, **os dados permanecem no MinIO**.
- **Tabela gerenciada (managed):** `CREATE TABLE ...` sem LOCATION — Spark gerencia dados e metadados. Ao executar `DROP TABLE`, **dados e metadados são removidos**.

Aqui usamos tabelas **não gerenciadas** pois os dados já existem no MinIO.

In [ ]:
tabelas_delta = [
    'categoria', 'fornecedor', 'produto', 'produto_fornecedor',
    'cliente', 'endereco', 'pedido', 'item_pedido', 'pagamento', 'avaliacao'
]

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

print('Tabelas registradas no Spark Catalog:')
spark.sql('SHOW TABLES').show(truncate=False)

## 3. Estado Inicial (SELECT)

In [ ]:
print('=== CATEGORIAS ===')
spark.sql('SELECT * FROM categoria ORDER BY cd_categoria').show()

print('=== PRODUTOS (primeiros 10) ===')
spark.sql('SELECT cd_produto, cd_categoria, nm_produto, vl_preco, qt_estoque FROM produto ORDER BY cd_produto LIMIT 10').show(truncate=False)

print('=== CLIENTES (primeiros 5) ===')
spark.sql('SELECT cd_cliente, nm_cliente, email, sexo, dt_nascimento FROM cliente LIMIT 5').show(truncate=False)

In [ ]:
# Contagem por tabela
print(f'{"Tabela":<25} {"Registros":>10}')
print('-' * 37)
for t in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {t}').collect()[0]['cnt']
    print(f'{t:<25} {count:>10}')

---
## 4. INSERT — Inserir Novos Registros

Inserimos novos produtos, uma nova categoria e novos clientes.

In [ ]:
# INSERT: nova categoria
print('--- INSERT: Nova categoria ---')
spark.sql('SELECT COUNT(*) as total FROM categoria').show()

spark.sql("""
    INSERT INTO categoria VALUES
    (11, 'GAMES E CONSOLES', 'Consoles jogos acessorios e perifericos gamer'),
    (12, 'PAPELARIA', 'Material escolar e de escritorio cadernos canetas')
""")

spark.sql('SELECT * FROM categoria ORDER BY cd_categoria').show()
print('2 novas categorias inseridas!')

In [ ]:
# INSERT: novos produtos nas novas categorias
print('--- INSERT: Novos produtos ---')

spark.sql("""
    INSERT INTO produto VALUES
    (51, 11, 'PLAYSTATION 5 SLIM', 'Console Sony PS5 Slim 1TB SSD leitor de disco', 4499.90, 30, '2024-01-15'),
    (52, 11, 'XBOX SERIES X', 'Console Microsoft Xbox Series X 1TB 4K HDR', 4299.90, 25, '2024-01-15'),
    (53, 11, 'NINTENDO SWITCH OLED', 'Console Nintendo Switch OLED tela 7 dock incluida', 2899.90, 40, '2024-02-01'),
    (54, 12, 'CADERNO UNIVERSITARIO 200FLS', 'Caderno espiral universitario 200 folhas capa dura', 39.90, 500, '2024-03-01'),
    (55, 12, 'KIT CANETAS 0.5MM 12 CORES', 'Kit canetas gel 0.5mm 12 cores escrita suave', 49.90, 300, '2024-03-01')
""")

spark.sql('SELECT cd_produto, cd_categoria, nm_produto, vl_preco FROM produto WHERE cd_produto >= 51 ORDER BY cd_produto').show(truncate=False)
print('5 novos produtos inseridos!')

In [ ]:
# INSERT: novos clientes
print('--- INSERT: Novos clientes ---')

spark.sql("""
    INSERT INTO cliente VALUES
    (901, 'JOAO DA SILVA TESTE', '12345678900', 'teste901@email.com.br', 'M', '1990-05-15', '2024-12-01'),
    (902, 'MARIA SOUZA TESTE', '98765432100', 'teste902@email.com.br', 'F', '1985-11-20', '2024-12-01'),
    (903, 'PEDRO ALVES EXEMPLO', '11122233344', 'teste903@email.com.br', 'M', '1992-07-08', '2024-12-02')
""")

spark.sql('SELECT * FROM cliente WHERE cd_cliente >= 900').show(truncate=False)
print('3 novos clientes inseridos!')

---
## 5. UPDATE — Atualizar Registros

Demonstramos UPDATE via **Spark SQL** e via **DeltaTable API**.

In [ ]:
# UPDATE via Spark SQL: reajuste de preço
print('--- UPDATE via SQL: Reajustar preço do PS5 ---')
print('ANTES:')
spark.sql('SELECT cd_produto, nm_produto, vl_preco FROM produto WHERE cd_produto = 51').show()

spark.sql("""
    UPDATE produto
    SET vl_preco = 3999.90
    WHERE cd_produto = 51
""")

print('DEPOIS:')
spark.sql('SELECT cd_produto, nm_produto, vl_preco FROM produto WHERE cd_produto = 51').show()

In [ ]:
# UPDATE via SQL: atualizar estoque
print('--- UPDATE via SQL: Atualizar estoque de múltiplos produtos ---')
print('ANTES (produtos 51-53):')
spark.sql('SELECT cd_produto, nm_produto, qt_estoque FROM produto WHERE cd_produto IN (51,52,53)').show()

spark.sql("""
    UPDATE produto
    SET qt_estoque = qt_estoque + 50
    WHERE cd_categoria = 11
""")

print('DEPOIS:')
spark.sql('SELECT cd_produto, nm_produto, qt_estoque FROM produto WHERE cd_produto IN (51,52,53)').show()

In [ ]:
# UPDATE via DeltaTable API
print('--- UPDATE via DeltaTable API: Corrigir e-mail do cliente 901 ---')
print('ANTES:')
spark.sql('SELECT cd_cliente, nm_cliente, email FROM cliente WHERE cd_cliente = 901').show(truncate=False)

dt_cliente = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/cliente')
dt_cliente.update(
    condition="cd_cliente = 901",
    set={"email": lit("joao.silva@empresareal.com.br"), "nm_cliente": lit("JOAO DA SILVA SANTOS")}
)

print('DEPOIS:')
spark.sql('SELECT cd_cliente, nm_cliente, email FROM cliente WHERE cd_cliente = 901').show(truncate=False)

---
## 6. DELETE — Remover Registros

Demonstramos DELETE via **Spark SQL** e via **DeltaTable API**.

In [ ]:
# DELETE via SQL: remover cliente de teste
print('--- DELETE via SQL: Remover clientes de teste ---')
print('ANTES:')
spark.sql('SELECT cd_cliente, nm_cliente FROM cliente WHERE cd_cliente >= 900').show()

spark.sql("DELETE FROM cliente WHERE cd_cliente = 903")

print('DEPOIS:')
spark.sql('SELECT cd_cliente, nm_cliente FROM cliente WHERE cd_cliente >= 900').show()

In [ ]:
# DELETE via SQL: remover categoria sem produtos
print('--- DELETE via SQL: Remover categoria PAPELARIA ---')
print('Produtos na categoria 12 antes do delete:')
spark.sql('SELECT cd_produto, nm_produto FROM produto WHERE cd_categoria = 12').show(truncate=False)

# Primeiro remove os produtos dependentes
spark.sql("DELETE FROM produto WHERE cd_categoria = 12")
# Depois remove a categoria
spark.sql("DELETE FROM categoria WHERE cd_categoria = 12")

print('Estado atual das categorias:')
spark.sql('SELECT * FROM categoria ORDER BY cd_categoria').show()

In [ ]:
# DELETE via DeltaTable API
print('--- DELETE via DeltaTable API: Remover produto com estoque zero ---')

# Verificar produtos com estoque zero (se houver)
spark.sql('SELECT cd_produto, nm_produto, qt_estoque FROM produto WHERE qt_estoque = 0').show(truncate=False)

dt_produto = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/produto')

# Forçar estoque zero em produto específico para demonstrar
dt_produto.update(condition="cd_produto = 52", set={"qt_estoque": lit(0)})

print('Antes do DELETE (produto 52):')
spark.sql('SELECT cd_produto, nm_produto, qt_estoque FROM produto WHERE cd_produto = 52').show()

dt_produto.delete("qt_estoque = 0 AND cd_produto = 52")

print('Após DELETE (produto 52 removido):')
spark.sql('SELECT cd_produto, nm_produto, qt_estoque FROM produto WHERE cd_produto >= 51').show(truncate=False)

---
## 7. HISTORY — Histórico de Versões Delta

O Delta Lake mantém um **transaction log** em `_delta_log/` que registra todas as operações.
O comando `DESCRIBE HISTORY` exibe o histórico completo.

In [ ]:
# Histórico da tabela produto
print('=== HISTÓRICO DA TABELA produto ===')
dt_produto = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/produto')
dt_produto.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

In [ ]:
# Histórico via Spark SQL
print('=== HISTÓRICO DA TABELA cliente ===')
spark.sql('DESCRIBE HISTORY cliente').select('version', 'timestamp', 'operation', 'operationParameters').show(truncate=False)

print('\n=== HISTÓRICO DA TABELA categoria ===')
spark.sql('DESCRIBE HISTORY categoria').select('version', 'timestamp', 'operation').show()

---
## 8. TIME TRAVEL — Leitura de Versões Anteriores

O Delta Lake permite ler qualquer versão histórica dos dados usando:
- `VERSION AS OF <n>` — por número de versão
- `TIMESTAMP AS OF '<timestamp>'` — por timestamp

In [ ]:
# Time Travel na tabela produto: versão original vs atual
print('=== produto — VERSÃO 0 (estado original do CSV) ===')
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/produto')
df_v0.orderBy('cd_produto').select('cd_produto','nm_produto','vl_preco','qt_estoque').show(5)
print(f'Registros na versão 0: {df_v0.count()}')

In [ ]:
# Versão atual
print('=== produto — VERSÃO ATUAL ===')
df_atual = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/produto')
df_atual.orderBy('cd_produto').select('cd_produto','nm_produto','vl_preco','qt_estoque').show(5)
print(f'Registros na versão atual: {df_atual.count()}')

In [ ]:
# Comparação: registros adicionados e removidos
print('=== Registros ADICIONADOS (existem no atual, não na v0) ===')
df_atual.subtract(df_v0).select('cd_produto','nm_produto','vl_preco').orderBy('cd_produto').show(truncate=False)

print('=== Registros REMOVIDOS (existem na v0, não no atual) ===')
df_v0.subtract(df_atual).select('cd_produto','nm_produto','vl_preco').orderBy('cd_produto').show(truncate=False)

In [ ]:
# Time Travel via Spark SQL
print('=== categoria — Time Travel via SQL ===')
spark.sql("SELECT * FROM categoria VERSION AS OF 0 ORDER BY cd_categoria").show()
print('\nVersão atual:')
spark.sql("SELECT * FROM categoria ORDER BY cd_categoria").show()

---
## 9. Resumo das Operações

In [ ]:
print('=' * 65)
print('RESUMO DAS OPERAÇÕES DML REALIZADAS — LojoDB')
print('=' * 65)
print()
print('INSERT:')
print('  - 2 novas categorias (GAMES E CONSOLES, PAPELARIA)')
print('  - 5 novos produtos (PS5, Xbox, Nintendo Switch, Caderno, Canetas)')
print('  - 3 novos clientes de teste')
print()
print('UPDATE:')
print('  - Reajuste de preço do PS5 via Spark SQL')
print('  - Reabastecimento de estoque de consoles (categoria 11) via SQL')
print('  - Correção de dados do cliente 901 via DeltaTable API')
print()
print('DELETE:')
print('  - Remoção do cliente 903 (teste) via Spark SQL')
print('  - Remoção da categoria PAPELARIA e produtos dependentes via SQL')
print('  - Remoção do produto com estoque zero via DeltaTable API')
print()
print('HISTORY e TIME TRAVEL:')
print('  - Histórico de transações via DeltaTable.history() e DESCRIBE HISTORY')
print('  - Leitura da versão 0 (estado original) via versionAsOf')
print('  - Comparação entre versões via subtract()')
print('=' * 65)

In [ ]:
spark.stop()
print('SparkSession finalizada.')